# Baseline Models Comparison

This notebook compares four baseline approaches for intent classification:
1. TF-IDF + Logistic Regression (sklearn)
2. Sentence Embeddings (all-MiniLM-L6-v2) + Logistic Regression
3. Hybrid Approach (TF-IDF + Embeddings concatenated) + Logistic Regression
4. Zero-shot with LLM API (Claude Haiku)

Metrics logged to wandb: Accuracy, Macro-F1, Per-class F1

## Setup and Dependencies

In [1]:
# Install required libraries
!pip install datasets transformers scikit-learn sentence-transformers wandb anthropic openai python-dotenv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.1 MB/s eta 0:00:0000:010:01


In [5]:
import os
from dotenv import load_dotenv

# Try to load from .env file (works locally)
load_dotenv()

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
WANDB_API_KEY = os.getenv('WANDB_API_KEY')

# If not found in .env, try Colab secrets
if not ANTHROPIC_API_KEY:
    try:
        from google.colab import userdata
        ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    except:
        pass

if not WANDB_API_KEY:
    try:
        from google.colab import userdata
        WANDB_API_KEY = userdata.get('WANDB_API_KEY')
    except:
        pass

# Verify keys are loaded
if not ANTHROPIC_API_KEY:
    print("Error: ANTHROPIC_API_KEY not found. Add it to Colab Secrets or .env file")
if not WANDB_API_KEY:
    print("Error: WANDB_API_KEY not found. Add it to Colab Secrets or .env file")


Error: ANTHROPIC_API_KEY not found. Add it to Colab Secrets or .env file
Error: WANDB_API_KEY not found. Add it to Colab Secrets or .env file


In [16]:
import os
import time
import numpy as np
import pandas as pd
from datasets import load_dataset, DatasetDict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

import wandb
from anthropic import Anthropic

In [17]:
# Initialize wandb
wandb.init(
    project="intent-classification-baselines",
    name="baseline-models-comparison",
    config={
        "tfidf_model": "sklearn-tfidf-logistic",
        "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
        "zeroshot_model": "claude-haiku",
    }
)

## Load and Prepare Dataset (Same as test_hf_pipeline.ipynb)

In [18]:
# Load the dataset
dataset = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset')
print(f"Dataset splits: {dataset.keys()}")
print(f"Train set size: {len(dataset['train'])}")

Dataset splits: dict_keys(['train'])
Train set size: 26872


In [19]:
# Use same split as test_hf_pipeline.ipynb
dataset['train'] = dataset['train'].shuffle(seed=42)

train_testvalid = dataset['train'].train_test_split(test_size=0.3, seed=42)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

split_dataset = DatasetDict({
    'train': train_testvalid['train'],
    'test': test_valid['test'],
    'validation': test_valid['train']
})

print(f"Train set size: {len(split_dataset['train'])}")
print(f"Validation set size: {len(split_dataset['validation'])}")
print(f"Test set size: {len(split_dataset['test'])}")

Train set size: 18810
Validation set size: 4031
Test set size: 4031


In [20]:
# Get label information
labels = split_dataset['train']['intent']
unique_labels = sorted(set(labels))
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

print(f"Number of classes: {len(unique_labels)}")
print(f"\nClasses: {unique_labels}")

Number of classes: 27

Classes: ['cancel_order', 'change_order', 'change_shipping_address', 'check_cancellation_fee', 'check_invoice', 'check_payment_methods', 'check_refund_policy', 'complaint', 'contact_customer_service', 'contact_human_agent', 'create_account', 'delete_account', 'delivery_options', 'delivery_period', 'edit_account', 'get_invoice', 'get_refund', 'newsletter_subscription', 'payment_issue', 'place_order', 'recover_password', 'registration_problems', 'review', 'set_up_shipping_address', 'switch_account', 'track_order', 'track_refund']


In [21]:
# Prepare data for training
X_train = split_dataset['train']['instruction']
y_train = np.array([label2id[label] for label in split_dataset['train']['intent']])

X_test = split_dataset['test']['instruction']
y_test = np.array([label2id[label] for label in split_dataset['test']['intent']])

X_val = split_dataset['validation']['instruction']
y_val = np.array([label2id[label] for label in split_dataset['validation']['intent']])

print(f"Training data shape: {len(X_train)}")
print(f"Test data shape: {len(X_test)}")
print(f"Validation data shape: {len(X_val)}")

Training data shape: 18810
Test data shape: 4031
Validation data shape: 4031


## Model 1: TF-IDF + Logistic Regression

In [ ]:
print("="*60)
print("MODEL 1: TF-IDF + Logistic Regression")
print("="*60)

# Training
start_time = time.time()
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

lr_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_model.fit(X_train_tfidf, y_train)
train_time = time.time() - start_time

print(f"\nTraining time: {train_time:.2f}s")

# Prediction
start_time = time.time()
y_pred_lr = lr_model.predict(X_test_tfidf)
inference_time = time.time() - start_time
tfidf_inference_time_per_sample = (inference_time / len(X_test)) * 1000  # ms per sample

# Metrics
accuracy_lr = accuracy_score(y_test, y_pred_lr)
macro_f1_lr = f1_score(y_test, y_pred_lr, average='weighted')
per_class_f1_lr = f1_score(y_test, y_pred_lr, average=None)

print(f"\nAccuracy: {accuracy_lr:.4f}")
print(f"Macro F1: {macro_f1_lr:.4f}")
print(f"Inference time per sample: {tfidf_inference_time_per_sample:.2f}ms")
print(f"Cost per 1k predictions: ~$0 (local inference)")
print(f"\nPer-class F1 Scores:")
for label, f1 in zip(unique_labels, per_class_f1_lr):
    print(f"  {label}: {f1:.4f}")

# Log to wandb
wandb.log({
    "tfidf_accuracy": accuracy_lr,
    "tfidf_macro_f1": macro_f1_lr,
    "tfidf_inference_time_ms": tfidf_inference_time_per_sample,
    "tfidf_cost_per_1k": 0.0
})

MODEL 1: TF-IDF + Logistic Regression

Training time: 6.73s

Accuracy: 0.9913
Macro F1: 0.9913
Inference time per sample: 0.00ms
Cost per 1k predictions: ~$0 (local inference)

Per-class F1 Scores:
  cancel_order: 0.9868
  change_order: 0.9739
  change_shipping_address: 0.9927
  check_cancellation_fee: 0.9925
  check_invoice: 0.9766
  check_payment_methods: 1.0000
  check_refund_policy: 0.9940
  complaint: 1.0000
  contact_customer_service: 0.9862
  contact_human_agent: 0.9836
  create_account: 0.9850
  delete_account: 0.9867
  delivery_options: 0.9962
  delivery_period: 0.9967
  edit_account: 0.9967
  get_invoice: 0.9817
  get_refund: 0.9937
  newsletter_subscription: 1.0000
  payment_issue: 0.9826
  place_order: 0.9930
  recover_password: 1.0000
  registration_problems: 0.9843
  review: 1.0000
  set_up_shipping_address: 0.9929
  switch_account: 1.0000
  track_order: 0.9932
  track_refund: 1.0000


## Model 2: Sentence Embeddings + Logistic Regression

In [ ]:
print("\n" + "="*60)
print("MODEL 2: Sentence Embeddings + Logistic Regression")
print("="*60)

# Load embedding model
print("\nLoading embedding model...")
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Encode texts
print("Encoding training texts...")
start_time = time.time()
X_train_emb = embedding_model.encode(list(X_train), batch_size=32, show_progress_bar=True)
X_test_emb = embedding_model.encode(list(X_test), batch_size=32, show_progress_bar=True)
encoding_time = time.time() - start_time

# Train logistic regression on embeddings
print("\nTraining Logistic Regression...")
start_time = time.time()
lr_emb_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_emb_model.fit(X_train_emb, y_train)
train_time = time.time() - start_time

print(f"Training time: {train_time:.2f}s")

# Prediction
start_time = time.time()
y_pred_emb = lr_emb_model.predict(X_test_emb)
inference_time = time.time() - start_time
embedding_inference_time_per_sample = (inference_time / len(X_test)) * 1000  # ms per sample

# Metrics
accuracy_emb = accuracy_score(y_test, y_pred_emb)
macro_f1_emb = f1_score(y_test, y_pred_emb, average='weighted')
per_class_f1_emb = f1_score(y_test, y_pred_emb, average=None)

print(f"\nAccuracy: {accuracy_emb:.4f}")
print(f"Macro F1: {macro_f1_emb:.4f}")
print(f"Inference time per sample: {embedding_inference_time_per_sample:.2f}ms")
print(f"Cost per 1k predictions: ~$0 (local inference)")
print(f"\nPer-class F1 Scores:")
for label, f1 in zip(unique_labels, per_class_f1_emb):
    print(f"  {label}: {f1:.4f}")

# Log to wandb
wandb.log({
    "embedding_accuracy": accuracy_emb,
    "embedding_macro_f1": macro_f1_emb,
    "embedding_inference_time_ms": embedding_inference_time_per_sample,
    "embedding_cost_per_1k": 0.0,
})


MODEL 2: Sentence Embeddings + Logistic Regression

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding training texts...


Batches:   0%|          | 0/588 [00:00<?, ?it/s]

Batches:   0%|          | 0/126 [00:00<?, ?it/s]


Training Logistic Regression...
Training time: 4.92s

Accuracy: 0.9943
Macro F1: 0.9943
Inference time per sample: 0.00ms
Cost per 1k predictions: ~$0 (local inference)

Per-class F1 Scores:
  cancel_order: 0.9902
  change_order: 0.9900
  change_shipping_address: 0.9963
  check_cancellation_fee: 1.0000
  check_invoice: 0.9864
  check_payment_methods: 1.0000
  check_refund_policy: 0.9941
  complaint: 1.0000
  contact_customer_service: 0.9966
  contact_human_agent: 0.9967
  create_account: 0.9970
  delete_account: 1.0000
  delivery_options: 0.9962
  delivery_period: 0.9967
  edit_account: 1.0000
  get_invoice: 0.9880
  get_refund: 0.9809
  newsletter_subscription: 1.0000
  payment_issue: 0.9931
  place_order: 0.9965
  recover_password: 1.0000
  registration_problems: 0.9906
  review: 0.9968
  set_up_shipping_address: 0.9930
  switch_account: 1.0000
  track_order: 0.9866
  track_refund: 0.9842


## Model 3: Zero-shot with LLM API (Claude Haiku)

In [ ]:
print("\n" + "="*60)
print("MODEL 3: Zero-shot Classification with Claude Haiku")
print("="*60)

# Initialize Anthropic client
client = Anthropic(api_key=ANTHROPIC_API_KEY)

# Use subset of test set for cost efficiency
test_sample_size = min(200, len(X_test))
X_test_sample = X_test[:test_sample_size]
y_test_sample = y_test[:test_sample_size]

print(f"\nUsing {test_sample_size} test samples for zero-shot evaluation")

# Create prompt template
intent_list = ", ".join(unique_labels)

# Classify samples
print("\nCalling Claude Haiku API...")
start_time = time.time()
y_pred_claude = []
total_input_tokens = 0
total_output_tokens = 0

for i, text in enumerate(X_test_sample):
    if i % 50 == 0:
        print(f"  Processed {i}/{test_sample_size} samples")

    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=50,
        messages=[
            {
                "role": "user",
                "content": f"""Classify the following customer support query into ONE of these intent categories:
{intent_list}

Query: {text}

Respond with ONLY the intent category name, nothing else."""
            }
        ]
    )

    response_text = message.content[0].text.strip().lower()
    # Find matching intent
    pred_intent = unique_labels[0]
    for intent in unique_labels:
        if intent.lower() in response_text:
            pred_intent = intent
            break

    y_pred_claude.append(label2id[pred_intent])
    total_input_tokens += message.usage.input_tokens
    total_output_tokens += message.usage.output_tokens

inference_time = time.time() - start_time
y_pred_claude = np.array(y_pred_claude)
claude_inference_time_per_sample = (inference_time / test_sample_size) * 1000  # ms per sample

# Calculate costs (Claude 3.5 Haiku pricing: $0.80 per 1M input, $4 per 1M output)
cost_per_input_1k = (0.80 / 1_000_000) * 1000
cost_per_output_1k = (4.00 / 1_000_000) * 1000
total_cost = (total_input_tokens * 0.80 / 1_000_000) + (total_output_tokens * 4.00 / 1_000_000)
cost_per_1k = ((total_input_tokens * 0.80 + total_output_tokens * 4.00) / 1_000_000) * 1000 / test_sample_size

# Metrics
accuracy_claude = accuracy_score(y_test_sample, y_pred_claude)
macro_f1_claude = f1_score(y_test_sample, y_pred_claude, average='weighted')
per_class_f1_claude = f1_score(y_test_sample, y_pred_claude, average=None)

print(f"\nAccuracy: {accuracy_claude:.4f}")
print(f"Macro F1: {macro_f1_claude:.4f}")
print(f"Inference time per sample: {claude_inference_time_per_sample:.2f}ms")
print(f"\nToken Usage:")
print(f"  Total input tokens: {total_input_tokens:,}")
print(f"  Total output tokens: {total_output_tokens:,}")
print(f"  Total cost: ${total_cost:.4f}")
print(f"  Cost per 1k predictions: ${cost_per_1k:.4f}")
print(f"\nPer-class F1 Scores:")
for label, f1 in zip(unique_labels, per_class_f1_claude):
    print(f"  {label}: {f1:.4f}")

# Log to wandb
wandb.log({
    "claude_accuracy": accuracy_claude,
    "claude_macro_f1": macro_f1_claude,
    "claude_inference_time_ms": claude_inference_time_per_sample,
    "claude_cost_per_1k": cost_per_1k,
    "claude_total_cost": total_cost,
    "claude_input_tokens": total_input_tokens,
    "claude_output_tokens": total_output_tokens
})


MODEL 3: Zero-shot Classification with Claude Haiku

Using 200 test samples for zero-shot evaluation

Calling Claude Haiku API...
  Processed 0/200 samples
  Processed 50/200 samples
  Processed 100/200 samples
  Processed 150/200 samples

Accuracy: 0.8600
Macro F1: 0.8548
Inference time per sample: 599.25ms

Token Usage:
  Total input tokens: 34,441
  Total output tokens: 1,351
  Total cost: $0.0330
  Cost per 1k predictions: $0.1648

Per-class F1 Scores:
  cancel_order: 0.8571
  change_order: 0.9091
  change_shipping_address: 0.8571
  check_cancellation_fee: 1.0000
  check_invoice: 0.8571
  check_payment_methods: 1.0000
  check_refund_policy: 1.0000
  complaint: 0.7368
  contact_customer_service: 0.6923
  contact_human_agent: 0.7692
  create_account: 0.9333
  delete_account: 1.0000
  delivery_options: 0.9091
  delivery_period: 0.8000
  edit_account: 0.7500
  get_invoice: 0.9630
  get_refund: 0.6667
  newsletter_subscription: 1.0000
  payment_issue: 1.0000
  place_order: 0.8571
  rec

## Model 3: Hybrid Approach (TF-IDF + Embeddings)

In [ ]:
print("\n" + "="*60)
print("MODEL 3: Hybrid Approach (TF-IDF + Embeddings)")
print("="*60)

# Combine TF-IDF and embedding features
print("\nCombining TF-IDF and embedding features...")
from scipy.sparse import hstack

# Concatenate features (sparse TF-IDF + dense embeddings)
X_train_hybrid = hstack([X_train_tfidf, X_train_emb])
X_test_hybrid = hstack([X_test_tfidf, X_test_emb])

print(f"Combined feature shape - Train: {X_train_hybrid.shape}, Test: {X_test_hybrid.shape}")

# Train logistic regression on hybrid features
print("\nTraining Logistic Regression on hybrid features...")
start_time = time.time()
lr_hybrid_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_hybrid_model.fit(X_train_hybrid, y_train)
train_time = time.time() - start_time

print(f"Training time: {train_time:.2f}s")

# Prediction
start_time = time.time()
y_pred_hybrid = lr_hybrid_model.predict(X_test_hybrid)
inference_time = time.time() - start_time
hybrid_inference_time_per_sample = (inference_time / len(X_test)) * 1000  # ms per sample

# Metrics
accuracy_hybrid = accuracy_score(y_test, y_pred_hybrid)
macro_f1_hybrid = f1_score(y_test, y_pred_hybrid, average='weighted')
per_class_f1_hybrid = f1_score(y_test, y_pred_hybrid, average=None)

print(f"\nAccuracy: {accuracy_hybrid:.4f}")
print(f"Macro F1: {macro_f1_hybrid:.4f}")
print(f"Inference time per sample: {hybrid_inference_time_per_sample:.2f}ms")
print(f"Cost per 1k predictions: ~$0 (local inference)")
print(f"\nPer-class F1 Scores:")
for label, f1 in zip(unique_labels, per_class_f1_hybrid):
    print(f"  {label}: {f1:.4f}")

# Log to wandb
wandb.log({
    "hybrid_accuracy": accuracy_hybrid,
    "hybrid_macro_f1": macro_f1_hybrid,
    "hybrid_inference_time_ms": hybrid_inference_time_per_sample,
    "hybrid_cost_per_1k": 0.0,
})

# Create comparison table
results_df = pd.DataFrame({
    'Model': ['TF-IDF + LR', 'Embeddings + LR', 'Hybrid (TF-IDF + Embeddings)', 'Claude Haiku (Zero-shot)'],
    'Accuracy': [accuracy_lr, accuracy_emb, accuracy_hybrid, accuracy_claude],
    'Macro F1': [macro_f1_lr, macro_f1_emb, macro_f1_hybrid, macro_f1_claude],
    'Inference Time (ms)': [
        tfidf_inference_time_per_sample,
        embedding_inference_time_per_sample,
        hybrid_inference_time_per_sample,
        claude_inference_time_per_sample
    ],
    'Cost per 1k': ['$0.00', '$0.00', '$0.00', f'${cost_per_1k:.4f}']
})

print("\n" + "="*80)
print("FINAL RESULTS COMPARISON")
print("="*80)
print(results_df.to_string(index=False))

# Log comparison to wandb
wandb.log({"results_comparison": wandb.Table(dataframe=results_df)})

In [ ]:
# Summary and recommendations
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"""
Best Accuracy: {results_df.loc[results_df['Accuracy'].idxmax(), 'Model']}
Best Macro F1: {results_df.loc[results_df['Macro F1'].idxmax(), 'Model']}
Fastest Inference: {results_df.loc[results_df['Inference Time (ms)'].idxmin(), 'Model']}
Lowest Cost: {results_df.loc[results_df['Cost per 1k'].idxmin(), 'Model']}

Recommendations:
- Best overall: {results_df.loc[results_df['Accuracy'].idxmax(), 'Model']} (highest accuracy, zero cost)
- For real-time requirements: TF-IDF + LR (fastest inference at <0.01ms per sample)
- For balanced performance: Hybrid approach combines both feature types with strong accuracy
- Zero-shot LLM: Good for few-shot scenarios but higher latency (~600ms) and cost ($0.16 per 1k)
""")

wandb.finish()

In [30]:
# Summary and recommendations
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"""
Best Accuracy: {results_df.loc[results_df['Accuracy'].idxmax(), 'Model']}
Best Macro F1: {results_df.loc[results_df['Macro F1'].idxmax(), 'Model']}
Fastest Inference: {results_df.loc[results_df['Inference Time (ms)'].idxmin(), 'Model']}
Lowest Cost: {results_df.loc[results_df['Cost per 1k'].idxmin(), 'Model']}

Recommendations:
- For production use with cost constraints: Use Embeddings + LR (good accuracy, zero cost)
- For best accuracy: Compare TF-IDF + LR and Embeddings + LR performance
- For real-time requirements: Both local models (TF-IDF and Embeddings) are sub-ms per sample
- Zero-shot LLM approach: Good for few-shot scenarios but higher latency and cost
""")

wandb.finish()


SUMMARY

Best Accuracy: Embeddings + LR
Best Macro F1: Embeddings + LR
Fastest Inference: TF-IDF + LR
Lowest Cost: TF-IDF + LR

Recommendations:
- For production use with cost constraints: Use Embeddings + LR (good accuracy, zero cost)
- For best accuracy: Compare TF-IDF + LR and Embeddings + LR performance
- For real-time requirements: Both local models (TF-IDF and Embeddings) are sub-ms per sample
- Zero-shot LLM approach: Good for few-shot scenarios but higher latency and cost



claude_accuracy,▁
claude_cost_per_1k,▁
claude_f1_cancel_order,▁
claude_f1_change_order,▁
claude_f1_change_shipping_address,▁
claude_f1_check_cancellation_fee,▁
claude_f1_check_invoice,▁
claude_f1_check_payment_methods,▁
claude_f1_check_refund_policy,▁
claude_f1_complaint,▁
+86,...
